# 02 - Verify the crops

Week 2, Task 2. Before any tracking work, each image has to earn its place.

Three questions:

1. **Where is the target?** Measured from the post-collapse image, not assumed.
2. **Is each image usable?** Enough texture on the slope to track, no snow blanket, no cloud.
3. **Which is the best base image** for the synthetic testbed?

The checks here are written as numbers so that the daily pipeline in week 8 can
reuse them. But look at the pictures too: the numbers were chosen by eye first,
and they are only as good as that judgment.

## 1. Load everything

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from scipy import ndimage

ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data" / "chamoli"
AFTER_DIR = DATA_DIR / "after"

index = pd.read_csv(DATA_DIR / "index.csv").sort_values("date").reset_index(drop=True)
imgs = {r["date"]: np.load(DATA_DIR / r["file"]) for _, r in index.iterrows()}
print(len(imgs), "pre-collapse images:", ", ".join(index["date"]))

after_files = sorted(AFTER_DIR.glob("*.npy"))
print("post-collapse images:", [f.stem for f in after_files])

## 2. Measure where the collapse actually happened

Not assumed. The scar is the patch that got darkest between the last pre-collapse
image and the first post-collapse one.

Use the **earliest** post-collapse image. Later ones have fresh snow over the scar.

In [ ]:
AFTER_DATE = "2021-02-10"          # change if you downloaded a different one
BEFORE_DATE = index["date"].iloc[-1]

after_img = np.load(AFTER_DIR / f"{AFTER_DATE}.npy")
before_img = imgs[BEFORE_DATE]


def normalise(a):
    a = a.astype("float64")
    return (a - np.nanmean(a)) / (np.nanstd(a) + 1e-9)


diff = normalise(after_img) - normalise(before_img)

DARKEN_PERCENTILE = 1              # strict: the real source was only ~0.25 km2
threshold = np.nanpercentile(diff, DARKEN_PERCENTILE)
darker = ndimage.binary_opening(diff < threshold, np.ones((3, 3)))

labels, n = ndimage.label(darker)
sizes = ndimage.sum(darker, labels, range(1, n + 1))
biggest = int(np.argmax(sizes)) + 1
rr, cc = np.where(labels == biggest)
SCAR = dict(r0=int(rr.min()), r1=int(rr.max()), c0=int(cc.min()), c1=int(cc.max()))
SCAR["row"] = (SCAR["r0"] + SCAR["r1"]) // 2
SCAR["col"] = (SCAR["c0"] + SCAR["c1"]) // 2

area_km2 = int(sizes[biggest - 1]) * 100 / 1e6
print(f"scar box: rows {SCAR['r0']}-{SCAR['r1']}, cols {SCAR['c0']}-{SCAR['c1']}")
print(f"centre:   row {SCAR['row']}, col {SCAR['col']}")
print(f"area:     {area_km2:.3f} km2   (published figure for the block: ~0.25 km2)")

### Is the target well inside the crop?

A tracker needs the moving patch surrounded by stable ground, so it can tell real
motion from a whole-image misalignment. A target hard against an edge is a problem.

In [ ]:
R, C = before_img.shape
margins = {
    "north (top)": SCAR["r0"],
    "south (bottom)": R - SCAR["r1"],
    "west (left)": SCAR["c0"],
    "east (right)": C - SCAR["c1"],
}
for side, px in margins.items():
    flag = "  <-- TIGHT" if px < 100 else ""
    print(f"{side:16s} {px:4d} px = {px * 10 / 1000:.2f} km{flag}")

if min(margins.values()) < 100:
    # Work out the box that would centre the target, for a re-download.
    BBOX = [79.705, 30.355, 79.757, 30.400]      # the box these crops came from
    lon = BBOX[0] + (SCAR["col"] / C) * (BBOX[2] - BBOX[0])
    lat = BBOX[3] - (SCAR["row"] / R) * (BBOX[3] - BBOX[1])
    hw, hh = (BBOX[2] - BBOX[0]) / 2, (BBOX[3] - BBOX[1]) / 2
    print(f"\nThe target is at about {lat:.4f} N, {lon:.4f} E.")
    print("A box centred on it would be:")
    print(f"  BBOX = [{lon - hw:.4f}, {lat - hh:.4f}, {lon + hw:.4f}, {lat + hh:.4f}]")
    print("Consider putting that in notebook 01 and downloading again.")

## 3. Quality score for each image

In [ ]:
PAD = 60        # look at the scar box plus this much around it


def box_slice(a, pad=PAD):
    return a[max(0, SCAR["r0"] - pad):SCAR["r1"] + pad,
             max(0, SCAR["c0"] - pad):SCAR["c1"] + pad]


def texture(a):
    """How much fine detail is there? Snow and deep shadow score near zero."""
    return float(np.nanstd(a - ndimage.gaussian_filter(a, 3)))


# A reference texture pattern: the median across all images, which averages out
# anything odd in a single date.
stack = np.stack([box_slice(img) for img in imgs.values()])
reference = np.median(stack, axis=0)
ref_hp = reference - ndimage.gaussian_filter(reference, 3)

rows = []
for date, img in imgs.items():
    b = box_slice(img)
    hp = b - ndimage.gaussian_filter(b, 3)
    rows.append(
        {
            "date": date,
            "texture": texture(b),
            "brightness": float(np.nanmedian(b) / (np.nanmedian(img) + 1e-9)),
            "valid_%": float(np.mean(np.isfinite(b) & (b > 0)) * 100),
            "match": float(np.corrcoef(hp.ravel(), ref_hp.ravel())[0, 1]),
        }
    )

q = pd.DataFrame(rows).merge(index[["date", "cloud", "collection"]], on="date")
q["texture_rel"] = q["texture"] / q["texture"].max()
q = q.sort_values("texture", ascending=False).reset_index(drop=True)
q.round(3)

What the columns mean:

- **texture** - fine detail on the slope. This is the one that matters most: no
  detail means nothing for the tracker to lock onto. `texture_rel` scales it
  against the best image in the set.
- **brightness** - the slope's brightness relative to the whole scene. Much above
  1 suggests snow cover.
- **valid_%** - pixels that carry real data.
- **match** - how well this image's texture agrees with the median of all of them.
  Low means something changed that shouldn't have: cloud, snow, or a bad scene.

In [ ]:
TEXTURE_MIN = 0.45      # fraction of the best image's texture
MATCH_MIN = 0.30        # correlation with the reference pattern

q["verdict"] = np.where(
    (q["texture_rel"] >= TEXTURE_MIN) & (q["match"] >= MATCH_MIN),
    "keep",
    "CHECK BY EYE",
)
print(q[["date", "texture_rel", "brightness", "match", "cloud", "verdict"]].round(3).to_string(index=False))
print()
flagged = q[q["verdict"] != "keep"]["date"].tolist()
print("flagged:", flagged if flagged else "none")
print("\nThresholds are a starting point. Look at the pictures below before trusting them.")

## 4. Look at the slope in every image

The table says which images to doubt. This says whether the table is right.
Each panel is the scar area plus a margin, with the measured scar box in red.

In [ ]:
n = len(q)
cols = 5
nrows = int(np.ceil(n / cols))
fig, axes = plt.subplots(nrows, cols, figsize=(4 * cols, 4.4 * nrows))
for ax in np.atleast_1d(axes).ravel():
    ax.axis("off")

for ax, (_, row) in zip(np.atleast_1d(axes).ravel(), q.iterrows()):
    b = box_slice(imgs[row["date"]])
    lo, hi = np.nanpercentile(b, [2, 98])
    ax.imshow(np.clip((b - lo) / (hi - lo + 1e-9), 0, 1), cmap="gray")
    ax.add_patch(
        Rectangle((PAD, PAD), SCAR["c1"] - SCAR["c0"], SCAR["r1"] - SCAR["r0"],
                  fill=False, ec="#ff3b30", lw=2)
    )
    ok = row["verdict"] == "keep"
    ax.set_title(f"{row['date']}\ntexture {row['texture_rel']:.2f}  match {row['match']:.2f}",
                 fontsize=10, color="black" if ok else "#ff3b30",
                 fontweight="normal" if ok else "bold")
plt.tight_layout()
plt.show()

Go through them one at a time:

- Can you see rock detail inside the red box, or is it a flat white or flat black area?
- If it is flat white, that is snow. If flat black, that is shadow. Either way the
  tracker has nothing to work with.
- Does anything look shifted compared to its neighbours? That is a georeferencing
  problem, and it is week 3's job to measure and correct.

Where your eye disagrees with the table, your eye wins, and the threshold should
move. Note down why, because that reasoning becomes the automatic quality check
in week 8.

## 5. Pick the base image for the testbed

The testbed needs one real image to build synthetic pairs from. The right choice is
the one with the most texture on the slope, since that is the hardest test of
whether the method can track small motion. It is not necessarily the least cloudy.

In [ ]:
kept = q[q["verdict"] == "keep"]
BASE_DATE = kept.iloc[0]["date"]       # highest texture among the kept images
print("base image:", BASE_DATE)
print(kept.iloc[0][["texture_rel", "brightness", "match", "cloud"]].round(3).to_string())

base = imgs[BASE_DATE]
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
lo, hi = np.nanpercentile(base, [2, 98])
axes[0].imshow(np.clip((base - lo) / (hi - lo), 0, 1), cmap="gray")
axes[0].add_patch(Rectangle((SCAR["c0"], SCAR["r0"]),
                            SCAR["c1"] - SCAR["c0"], SCAR["r1"] - SCAR["r0"],
                            fill=False, ec="#ff3b30", lw=2))
axes[0].set_title(f"base image: {BASE_DATE}")

b = box_slice(base, pad=30)
lo, hi = np.nanpercentile(b, [2, 98])
axes[1].imshow(np.clip((b - lo) / (hi - lo), 0, 1), cmap="gray")
axes[1].set_title("the patch the testbed will move")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Save the results

In [ ]:
q.to_csv(DATA_DIR / "quality.csv", index=False)

settings = {
    "base_date": BASE_DATE,
    "after_date": AFTER_DATE,
    "scar_r0": SCAR["r0"], "scar_r1": SCAR["r1"],
    "scar_c0": SCAR["c0"], "scar_c1": SCAR["c1"],
    "scar_row": SCAR["row"], "scar_col": SCAR["col"],
    "rejected": ";".join(flagged),
}
pd.Series(settings).to_csv(DATA_DIR / "settings.csv", header=False)

print("wrote quality.csv and settings.csv")
for k, v in settings.items():
    print(f"  {k}: {v}")
print("\nNotebook 03 (the testbed) reads settings.csv, so the scar box and base")
print("image are defined in one place.")